# 02 - Model Comparison
## Menjalankan 3 Model: TF-IDF, SBERT, IndoBERT

In [ ]:
!pip install -q scikit-learn sentence-transformers transformers torch pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

df_test = pd.read_csv('data/msrp_val_clean.csv')
print(f"Test set: {len(df_test)} pairs")
print(f"Label distribution:\n{df_test['label'].value_counts()}")

pairs = list(zip(df_test['sentence1_clean'], df_test['sentence2_clean']))
y_true = df_test['label'].values

## Model 1: TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Running TF-IDF...")
vectorizer = TfidfVectorizer(max_features=10000)
all_texts = [t for pair in pairs for t in pair]
vectorizer.fit(all_texts)

texts1, texts2 = zip(*pairs)
vec1 = vectorizer.transform(texts1)
vec2 = vectorizer.transform(texts2)
sims_tfidf = cosine_similarity(vec1, vec2).diagonal()

print(f"TF-IDF done. Mean similarity: {sims_tfidf.mean():.4f}")

## Model 2: SBERT

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading SBERT...")
sbert_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

print("Encoding with SBERT...")
emb1_sbert = sbert_model.encode(texts1, batch_size=32, show_progress_bar=True)
emb2_sbert = sbert_model.encode(texts2, batch_size=32, show_progress_bar=True)

sims_sbert = np.array([cosine_similarity([e1], [e2])[0][0] for e1, e2 in zip(emb1_sbert, emb2_sbert)])
print(f"SBERT done. Mean similarity: {sims_sbert.mean():.4f}")

## Model 3: IndoBERT

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("Loading IndoBERT...")
tokenizer = AutoTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
indobert_model = AutoModel.from_pretrained('indobenchmark/indobert-base-p1').to(device)
indobert_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def encode_indobert(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            model_output = indobert_model(**encoded)
        batch_emb = mean_pooling(model_output, encoded['attention_mask'])
        embeddings.append(batch_emb.cpu().numpy())
    return np.vstack(embeddings)

print("Encoding with IndoBERT...")
emb1_indobert = encode_indobert(texts1)
emb2_indobert = encode_indobert(texts2)

sims_indobert = np.array([cosine_similarity([e1], [e2])[0][0] for e1, e2 in zip(emb1_indobert, emb2_indobert)])
print(f"IndoBERT done. Mean similarity: {sims_indobert.mean():.4f}")

## Simpan Hasil

In [ ]:
os.makedirs('data', exist_ok=True)

results = {
    'y_true': y_true,
    'sims_tfidf': sims_tfidf,
    'sims_sbert': sims_sbert,
    'sims_indobert': sims_indobert
}

with open('data/similarities.pkl', 'wb') as f:
    pickle.dump(results, f)

print("Saved: data/similarities.pkl")
print("\nLanjut ke 03_evaluation_final.ipynb")